# Analyse Water Storage: Surface Waterbodies

Inspect individual waterbodies, seasonal water extent and the tehsil’s total mapped water area.

Run the cells in order. Change the place, identifier or columns to explore other records. Downloads from GeoLibre use your selected tehsil; these templates start with Hilsa, Nalanda, Bihar.


## Set up Python

Run the collapsed setup cells. They import the libraries and define `read_json`, a small response reader. It reads JSON text, treats non-standard `NaN` and `Infinity` numbers as missing, and also accepts JSON returned inside a string. HTTP errors and malformed responses remain visible. Expand the cells to read the code.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geopandas", "matplotlib", "requests", "pyodide-http"])
    import pyodide_http
    pyodide_http.patch_all()

import os
import re
import ast
import json
from getpass import getpass
from inspect import isawaitable
from urllib.parse import urljoin
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, FileLink
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})


In [ ]:
SCOPE = json.loads("{\"state\": \"Bihar\", \"district\": \"Nalanda\", \"tehsil\": \"Hilsa\"}")
API_URL = 'https://geoserver.core-stack.org/api/v1/'
STAC_URL = 'https://spatio-temporal-asset-catalog.s3.ap-south-1.amazonaws.com/CorestackCatalogs_merged_collection/tehsil_wise/catalog.json'
YEARS = list(range(2017, 2025))


In [ ]:
"""Small response reader embedded in the notebooks' collapsed setup cell."""
import json


def read_json(response):
    """Read JSON text; represent non-standard NaN/Infinity values as missing."""
    response.raise_for_status()
    raw_text = response.text.lstrip("\ufeff")
    try:
        # Some API tables contain bare NaN or Infinity, which are not JSON numbers.
        data = json.loads(raw_text, parse_constant=lambda value: None)
        # Also accept a JSON document returned as a JSON-encoded string.
        if isinstance(data, str):
            data = json.loads(data.lstrip("\ufeff"), parse_constant=lambda value: None)
        return data
    except ValueError as error:
        raise ValueError(
            "The server response is not readable JSON. "
            "Inspect response.status_code and response.text[:500], then retry the request."
        ) from error


## Choose the place and set your API key

Edit `SCOPE` in the setup cell to change the place. The [public API guide](https://docs.core-stack.org/use-precomputed-data/public-apis/) explains registration and API keys. This cell reuses `CORE_STACK_API_KEY` or asks for it privately, then stores it in this kernel’s environment. The key is sent only to the API, in the `X-API-Key` header. Restart the kernel and run from the top after changing places.


In [ ]:
place = {key: re.sub(r"[\s_]+", "_", SCOPE[key].replace("(", "").replace(")", "")).strip("_").lower()
         for key in ["state", "district", "tehsil"]}
state, district, tehsil = place["state"], place["district"], place["tehsil"]
api_key = os.environ.get("CORE_STACK_API_KEY", "").strip()
if not api_key:
    api_key = getpass("CoRE Stack API key: ")
    if isawaitable(api_key):
        api_key = await api_key
os.environ["CORE_STACK_API_KEY"] = str(api_key).strip()
api_headers = {"X-API-Key": os.environ["CORE_STACK_API_KEY"]}
display(place)


## Discover data and descriptions in STAC

STAC lists published datasets, field descriptions, downloads and styles. Change `dataset` to another item from the collection. Asset links are used as published, wherever the files are hosted. STAC describes asset fields; API tables may use different names and units, which are shown explicitly in the examples below.


In [ ]:
collection_url = urljoin(STAC_URL, f"{state}/{district}/{tehsil}/collection.json")
response = requests.get(collection_url, timeout=90)
collection = read_json(response)
items = pd.DataFrame([{"Item": link["href"].split("/")[-1].removesuffix(".json"),
                       "URL": urljoin(collection_url, link["href"])}
                      for link in collection["links"] if link["rel"] == "item"], columns=["Item", "URL"])
# Follow a relevant item link from the collection.
dataset = "surface_water_bodies_vector"
matches = items.loc[items["Item"].str.endswith("_" + dataset)]
item = None
if not matches.empty:
    item_url = matches.iloc[0]["URL"]
    response = requests.get(item_url, timeout=90)
    item = read_json(response)
    display(Markdown(item["properties"].get("description", "No description published.")))
    field_notes = pd.DataFrame(item["properties"].get("table:columns", []))
    display(field_notes.reindex(columns=["name", "type", "description"]).head(12))
    print("Published field count:", len(field_notes), "— use field_notes to see them all.")
    display(pd.DataFrame(item["assets"]).T.reindex(columns=["title", "type", "href"]))
else:
    print("This dataset is not listed in the tehsil's STAC collection. Available items:")
    display(items)


## Read the STAC asset and choose a waterbody

Use the GeoJSON download published by this STAC item. The same cell lists identifiers and selects the first waterbody. Change `waterbody_id` to explore another one.


In [ ]:
waterbodies = gpd.GeoDataFrame()
waterbody = pd.Series(dtype=object)
if item is not None:
    asset_url = urljoin(item_url, item["assets"]["data"]["href"])
    response = requests.get(asset_url, timeout=180)
    waterbodies = gpd.GeoDataFrame.from_features(read_json(response)["features"], crs="EPSG:4326")
    display(waterbodies[["UID"]])
    waterbody_id = str(waterbodies.iloc[0]["UID"])
    waterbody = waterbodies.loc[waterbodies["UID"].astype(str) == waterbody_id].iloc[0]
    display(waterbody.drop(labels="geometry").iloc[:12].to_frame("First 12 fields"))


## Water area across years and seasons

Annual `area_YY-YY` values are hectares. Seasonal `k_`, `kr_` and `krz_` values are percentages of the `area_ored` reference footprint, also in hectares. Multiply that footprint by the seasonal percentage divided by 100. Missing years remain blank.


In [ ]:
if not waterbody.empty:
    area = pd.DataFrame({"Annual": [waterbody.get(f"area_{y%100:02d}-{(y+1)%100:02d}") for y in YEARS]}, index=YEARS)
    footprint_ha = pd.to_numeric(waterbody.get("area_ored"), errors="coerce")
    for prefix, season in [("k", "Kharif"), ("kr", "Rabi"), ("krz", "Zaid")]:
        shares = pd.to_numeric(pd.Series([waterbody.get(f"{prefix}_{y%100:02d}-{(y+1)%100:02d}") for y in YEARS], index=YEARS), errors="coerce")
        area[season] = footprint_ha * shares / 100
    area = area.apply(pd.to_numeric, errors="coerce")
    display(area.rename_axis("Starting year").round(3))
    area.plot(figsize=(10, 4), marker="o", ylabel="Water area (ha)", title=f"Water extent · {waterbody_id}")
    plt.tight_layout()
    plt.show()


## Total mapped water area through time

Sum annual areas across the waterbody features in this STAC asset. This is water extent, not storage volume. The record count helps identify years with incomplete reporting; missing areas are not filled with zero.


In [ ]:
if not waterbodies.empty:
    year_fields = [f"area_{y%100:02d}-{(y+1)%100:02d}" for y in YEARS]
    annual_areas = waterbodies.reindex(columns=year_fields).apply(pd.to_numeric, errors="coerce")
    annual_areas.columns = YEARS
    totals = pd.DataFrame({"Water area (ha)": annual_areas.sum(min_count=1), "Records with area": annual_areas.count()})
    display(totals)
    totals["Water area (ha)"].plot(figsize=(9, 3), marker="o", ylabel="Water area (ha)", title=f"Total mapped water area · {tehsil}")
    plt.tight_layout()
    plt.show()


## Explore the waterbody API

The API has its own waterbody inventory. Select an identifier returned by `get_waterbodies_data_by_admin` before calling `get_waterbody_data`; do not assume it matches the STAC asset’s identifiers. The response lists whichever property groups are available.


In [ ]:
response = requests.get(API_URL + "get_waterbodies_data_by_admin/", params=place, headers=api_headers, timeout=180)
inventory = read_json(response)
display(pd.DataFrame({"API waterbody ID": list(inventory)}))
api_waterbody = {}
if inventory:
    api_waterbody_id = next(iter(inventory))
    response = requests.get(API_URL + "get_waterbody_data/", params={**place, "uid": api_waterbody_id}, headers=api_headers, timeout=180)
    api_waterbody = read_json(response)[api_waterbody_id]
    display(pd.DataFrame({"Property group": list(api_waterbody), "Fields": [list(value) if isinstance(value, dict) else value for value in api_waterbody.values()]}))


## Inspect a property group

Change `property_group` to another name above. Zone-of-influence NDVI describes vegetation around the waterbody, not water area. Its yearly fields contain date/value JSON that can be read as a time series.


In [ ]:
property_group = "zoi_properties"
properties = api_waterbody.get(property_group, {})
display(pd.Series(properties, dtype=object).head(12).to_frame("First 12 fields"))
ndvi = {}
for year in YEARS:
    values = properties.get(f"NDVI_{year}")
    if values:
        ndvi.update(json.loads(values, parse_constant=lambda value: None) if isinstance(values, str) else values)
if ndvi:
    vegetation = pd.to_numeric(pd.Series(ndvi), errors="coerce")
    vegetation.index = pd.to_datetime(vegetation.index)
    vegetation.sort_index().plot(figsize=(10, 3), ylabel="NDVI (unitless)", ylim=(-1, 1), title=f"Zone-of-influence vegetation · {api_waterbody_id}")
    plt.tight_layout()
    plt.show()
